In [ ]:
# === 0. SETUP =============================================================# Funciona en local (desde el repo) y en Google Colab (clona si hace falta).import importlib.util, subprocess, sys, pathlibREPO = "https://github.com/JoMZ-ops/Non-Local-AID-thesis"RAMA = "claude/nonlocal-memory-system-analysis-t9dnw3"if importlib.util.find_spec("nlaid") is None:    raiz = pathlib.Path.cwd()    if not (raiz / "nlaid").is_dir():        raiz = raiz.parent    if (raiz / "nlaid").is_dir():        sys.path.insert(0, str(raiz))          # local: repo en el directorio padre    else:        subprocess.run(["pip", "install", "-q", "numpy", "scipy", "matplotlib"])        subprocess.run(["git", "clone", "-q", "--branch", RAMA, REPO])        raiz = pathlib.Path("Non-Local-AID-thesis")        sys.path.insert(0, str(raiz))else:    raiz = pathlib.Path(importlib.util.find_spec("nlaid").origin).parent.parentRAIZ = pathlib.Path(raiz)print("raiz:", RAIZ.resolve())

In [ ]:
import json, mathimport numpy as npimport matplotlib.pyplot as pltnp.set_printoptions(precision=6)plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,                     "grid.alpha": .3, "font.size": 10})from nlaid.core import Params, minkowski_dot, make_regulatorfrom nlaid.block1_linear import (numerator_N, susceptibility,                                 susceptibility_small_omega,                                 count_zeros_uhp, find_zeros_uhp,                                 dominant_pole, critical_cutoff)from nlaid.worldline import rest_history, unit_normal, smooth_bumpfrom nlaid.block2_delay import rhs_delay, integrate_delayfrom nlaid.block3_memory import rhs_memory, integrate_memoryfrom nlaid.block4_renorm import relaxation_rate, linearized_rate, stability_edge

In [ ]:
# === 1. core.py: convenciones y metrica ===================================# Signatura (+,-,-,-), c = 1, r0 = 1, tiempo propio con xdot^2 = 1.# Los dos parametros libres son los ejes de la Fig. 2 del paper.p = Params(ell=1/3, m_over_mB=2.0)          # r0/ell = 3 (caso de la Fig. 1)print(p, "  r0/ell =", p.r0_over_ell)v = np.array([np.cosh(0.7), np.sinh(0.7)])   # 4-velocidad temporal unitariaprint("xdot^2 =", minkowski_dot(v, v))

In [ ]:
# --- Reguladores: las tres condiciones del paper (p. 6) ------------------#   (i)   int dz delta_B(z) = 1        preserva el flujo radiado#   (ii)  delta_B(0) = 0               separa los puntos singulares#   (iii) delta_B(z) = 0 para z < 0    suprime la interaccion superluminicaell = 0.4suav = make_regulator("smeared", ell)        # ec. (5)desp = make_regulator("shifted", ell)        # ec. (4)for k, val in suav.check_conditions().items():    print(f"{k:22s} {val}")z = np.linspace(-0.2, 6*ell**2, 600)plt.plot(z, suav.delta(z), lw=2)plt.axvline(0, color="k", lw=.8)plt.xlabel("$z = x^2$"); plt.ylabel(r"$\delta_B$")plt.title(f"Regulador suavizado, ec. (5), $\\ell$ = {ell}")plt.show()

In [ ]:
# --- Momentos contra las formas cerradas ---------------------------------# Derivadas a mano para validar el codigo contra valores exactos, no contra# otra corrida del propio codigo.#   suavizado:  delta_m/m = r0/(6 ell)   I2 = 2 ell#   desplazado: delta_m/m = r0/(2 ell)   I2 = ell/2print(f"{'':12s} {'delta_m/m':>12} {'exacto':>12} | {'I2':>10} {'exacto':>10}")for nom, reg, dm, i2 in (("suavizado",  suav, 1/(6*ell), 2*ell),                         ("desplazado", desp, 1/(2*ell), ell/2)):    print(f"{nom:12s} {reg.mass_shift_over_m():12.8f} {dm:12.8f} | "          f"{reg.moment_u2():10.6f} {i2:10.6f}")# m_B = m - delta_m cambia de signo cuando delta_m = m:#   r0/ell = 2 (desplazado), 6 (suavizado) -> asintotas verticales de m/m_Bfor kind, nom, xc in (("shifted", "desplazado", 2.0), ("smeared", "suavizado", 6.0)):    r = make_regulator(kind, 1/xc)    print(f"{nom:12s} en r0/ell={xc}:  delta_m/m = {r.mass_shift_over_m():.6f}"          f"   ->  m_B = 0")

In [ ]:
# === 2. block1_linear.py: susceptibilidad, ec. (14) =======================# chi^r_w = 1 + r0[ (2/3) i w - (2/w^2) int du delta_B(u^2) N(w u)/u^2 ]# con N(phi) = (1 + i phi - phi^2) e^{-i phi} - 1 + phi^2/2 - (2/3) i phi^3.## Los cuatro primeros ordenes de N se cancelan identicamente, dejando#     N(phi) = sum_{n>=4} (-i phi)^n (n-1)^2 / n!# El orden dominante (3/8) phi^4 coincide con lo que afirma el paper (p. 8):# esa es la verificacion de que la ec. (14) esta bien transcrita.## La forma DIRECTA es inutilizable a phi pequeno por cancelacion catastrofica:phis = np.logspace(-6, -1, 40)directa = ((1 + 1j*phis - phis**2)*np.exp(-1j*phis) - 1           + 0.5*phis**2 - (2/3)*1j*phis**3)exacto = (3/8)*phis**4serie = np.array([numerator_N(f).real for f in phis])plt.loglog(phis, np.abs(directa.real - exacto)/exacto, "o-", label="forma directa")plt.loglog(phis, np.abs(serie - exacto)/exacto + 1e-17, "s-", label="serie (el codigo)")plt.xlabel(r"$\phi$"); plt.ylabel(r"error relativo vs $(3/8)\phi^4$")plt.legend(); plt.title("Cancelacion catastrofica en el numerador de la ec. (14)")plt.show()

In [ ]:
# --- chi y sus dos controles ---------------------------------------------pe = Params(ell=0.4)w = np.linspace(0.01, 3, 300)chi = susceptibility(w, suav, pe)fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))ax[0].plot(w, chi.real, label="Re"); ax[0].plot(w, chi.imag, label="Im")ax[0].set_xlabel(r"$\omega$"); ax[0].set_title(r"$\chi^r_\omega$ (suavizado)"); ax[0].legend()# control 1: desarrollo chi = 1 + r0[(2/3) i w - (3/4) w^2 I2] + O(w^3)ws = np.linspace(1e-3, .3, 80)ax[1].loglog(ws, np.abs(susceptibility(ws, suav, pe)                        - susceptibility_small_omega(ws, suav, pe)))ax[1].set_xlabel(r"$\omega$"); ax[1].set_title(r"$|\chi-\chi_{serie}|$ (debe ir como $\omega^3$)")plt.tight_layout(); plt.show()# control 2: forma cerrada EXACTA del regulador desplazado, porque# int_{-inf}^0 du delta_B(u^2) f(u) = f(-ell)/(2 ell)wt = np.array([0.2, 1.0, 3.0, 8.0 + 0.3j])cerrada = 1 + pe.r0*((2/3)*1j*wt - numerator_N(-wt*pe.ell)/(wt**2*pe.ell**3))print("max |numerico - cerrado| =", np.max(np.abs(susceptibility(wt, desp, pe) - cerrada)))

In [ ]:
# --- Ceros de chi = polos de F^r = polos runaway --------------------------# F^r_w = 1/[(w+ie)^2 chi^r_w], asi que los polos de F^r son los CEROS de chi.# Estable y causal <=> sin ceros en Im(w) > 0.# Al remover el cutoff, chi -> 1 + (2/3) i r0 w (ec. 15), cuyo cero esta en# w = 3i/2: la autoaceleracion de Abraham-Lorentz.ell_p = 0.02regp, pp = make_regulator("smeared", ell_p), Params(ell=ell_p)kw = dict(re_max=6.0, im_hi=6.0)# El conteo por principio del argumento es global (no depende de semillas);# la localizacion por Muller es local. Que coincidan es el control.print("principio del argumento:", count_zeros_uhp(regp, pp, n=600, **kw))zs = find_zeros_uhp(regp, pp, grid=120, **kw)print("Muller                 :", len(zs))for zz in zs:    print(f"   omega = {zz.real:+.5f} {zz.imag:+.5f}i     (ec. 15 predice 1.5i)")

In [ ]:
# --- Cutoff critico: donde el cero entra al semiplano superior ------------for kind in ("shifted", "smeared"):    xc = critical_cutoff(kind, lo=0.3, hi=30.0, tol=5e-3, n=800,                         re_max=12.0, im_hi=12.0)    print(f"{kind:9s}  r0/ell critico = {xc:.3f}   (ell_c = {1/xc:.3f} r0)")

In [ ]:
# === 3. worldline.py: la historia compartida por los bloques 2 y 3 ========# Una carga en reposo es solucion EXACTA de las tres ecuaciones: el vector# V1 = (x-x')(xdot.xdot') - [xdot.(x-x')] xdot' se anula identicamente para# movimiento inercial. Por eso la prehistoria inercial no mete transitorios.wl = rest_history(dim=2, s_rest=10.0, ds=1e-3)print("prehistoria en reposo: max |xdot^2 - 1| =", wl.norm_drift.max())# La excitacion es la fuente externa k^mu de la ec. (2), un pulso C^infinito.# Prescribir una trayectoria y apagarla de golpe haria saltar xddot en s=0;# en una ecuacion con retardo esa discontinuidad se propaga a s = ell, 2ell...# y degrada el orden del integrador de 2 a ~1.2.for th in (0.0, 0.4, -1.2):    vv = np.array([np.cosh(th), np.sinh(th)]); nn = unit_normal(vv)    print(f"  theta={th:+.1f}  n.v={minkowski_dot(nn,vv):+.1e}  n.n={minkowski_dot(nn,nn):+.6f}")# En reposo el punto retardado esta exactamente a distancia ell:for L in (0.1, 0.37, 1.0):    sp, *_ = wl.retarded_point(wl.s_max, wl._x[-1], L)    print(f"  ell={L:4.2f}  retardo s-s' = {wl.s_max - sp:.10f}")

In [ ]:
# === 4-5. Ortogonalidad: el control mas sensible ==========================# Contrayendo los lados derechos de las ecs. (6), (16) y (17) con xdot_mu se# obtiene CERO identicamente. Luego xdot^2 = 1 no es una restriccion a# imponer sino una consecuencia exacta, y su deriva mide el error de# integracion gratis. Si una transcripcion estuviera mal, esto lo detecta.pd = Params(ell=0.3, m_over_mB=0.5)wl2 = integrate_delay(pd, s_end=2.0, ds=2e-3)                 # ec. (16)x, v = wl2._x[-1], wl2._v[-1]_, xp, vp, ap = wl2.retarded_point(wl2.s_max + 1e-9, x, pd.ell)acc2, _ = rhs_delay(x, v, xp, vp, ap, pd)print(f"ec. (16):  xddot.xdot = {minkowski_dot(acc2, v):+.3e}   |xddot| = {np.linalg.norm(acc2):.4f}")wl3 = integrate_memory(pd, s_end=2.0, ds=2e-3, n_ell=20.0)    # ec. (17)acc3 = rhs_memory(wl3._x[-1], wl3._v[-1], wl3, wl3.s_max, pd, n_ell=20.0)print(f"ec. (17):  xddot.xdot = {minkowski_dot(acc3, wl3._v[-1]):+.3e}   |xddot| = {np.linalg.norm(acc3):.4f}")

In [ ]:
# --- Convergencia de orden 2 en ds (ec. 16) ------------------------------ (~1 min)probe = np.linspace(1.5, 4.0, 40)sols = {d: integrate_delay(Params(ell=0.5, m_over_mB=0.4), s_end=4.0, ds=d)        for d in (2e-2, 1e-2, 5e-3, 2.5e-3)}ref = sols[2.5e-3].sample_many(probe)[0]errs = [np.max(np.abs(sols[d].sample_many(probe)[0] - ref)) for d in (2e-2, 1e-2, 5e-3)]print(f"{'ds':>8} {'error':>12} {'orden':>7}")for i, d in enumerate((2e-2, 1e-2, 5e-3)):    o = "" if i == 0 else f"{math.log2(errs[i-1]/errs[i]):.2f}"    print(f"{d:8.4f} {errs[i]:12.3e} {o:>7}")print("deriva:", {d: f"{s.norm_drift.max():.1e}" for d, s in sols.items()})

In [ ]:
# --- Las TRES discretizaciones de la ec. (17) convergen por separado ------pm = Params(ell=0.5, m_over_mB=0.4)for etiqueta, kwargs in (("n_ell (ventana de memoria)",                          [dict(n_ell=n, pts_per_ell=16) for n in (10, 20, 30, 40)]),                         ("pts_per_ell (cuadratura)",                          [dict(n_ell=25, pts_per_ell=q) for q in (4, 8, 16, 32)])):    print(etiqueta)    prev = None    for kwa in kwargs:        a = np.linalg.norm(integrate_memory(pm, s_end=2.0, ds=1e-2, **kwa)._a[-1])        d = "" if prev is None else f"   cambio = {abs(a-prev):.2e}"        print(f"   {kwa}  |xddot| = {a:.12f}{d}")        prev = a

In [ ]:
# === 6. block4_renorm.py: la prueba cruzada ==============================# La ec. (18), chi = 1 + (2/3) i r0 w, NO puede imponerse literalmente: su# unico cero esta en w = 3i/2, en el semiplano SUPERIOR. Una teoria que la# satisfaga exactamente no tiene modo de relajacion que monitorear.## linearized_rate viene del BLOQUE 1 (espectral).# relaxation_rate viene del BLOQUE 3 (integracion temporal no lineal).# No comparten codigo mas alla de core: que coincidan valida ambos.pc = Params(ell=1/3, m_over_mB=2.0)lin = linearized_rate(pc)nolin, diag = relaxation_rate(pc, s_end=14.0, ds=5e-3)print(f"bloque 1 (espectral) : {lin:+.5f}")print(f"bloque 3 (no lineal) : {nolin:+.5f}")print(f"diferencia relativa  : {abs(lin-nolin)/abs(lin):.2%}")print(f"deriva xdot^2        : {diag['drift']:.1e}  (fiable={diag['fiable']})")# El contraterm de la ec. (10) da exactamente 2.0 a r0/ell = 3, que es el# valor central que la Fig. 1(a) del paper bracketea con 1.95 y 1.98.print("contraterm:", make_regulator("smeared", 1/3).m_over_mB_counterterm())

In [ ]:
# --- Ley del borde de estabilidad ----------------------------------------# La frontera NO cae a m/m_B fijo sino a r_0B/ell constante, con# r_0B = r0 (m/m_B) el radio clasico DESNUDO: el criterio compara el# acoplamiento desnudo con el cutoff, no con r0.sm = json.load(open(RAIZ/"data/borde_smeared.json"))print(f"{'r0/ell':>7} {'rama +':>9} {'rama -':>9} | {'x(m/mB)+':>10} {'x(m/mB)-':>10}")for k, val in sorted(sm.items(), key=lambda kv: float(kv[0])):    x = float(k); f = lambda q: float("nan") if q is None else q    P, N = f(val.get("positiva")), f(val.get("negativa"))    print(f"{x:7.1f} {P:9.3f} {N:9.3f} | {x*P:10.3f} {x*N:10.3f}")for rama in ("positiva", "negativa"):    vv = [float(k)*val[rama] for k, val in sm.items() if val.get(rama)]    print(f"r_0B/ell rama {rama:9s}: {np.mean(vv):+8.3f} +- {np.std(vv):.3f}  ({len(vv)} pts)")

In [ ]:
# === 7. Comparacion con la Fig. 2 del paper ==============================# data/fig2_digitalizada.npz: frontera extraida del PDF por pixeles.# Calibracion en el .json: eje y=0 en la fila 446, 367 px/unidad en y,# 59.36 px/unidad de r0/ell (reticula de marcas menores confirmada).## HALLAZGO (docs/discrepancias.md, F1): la linea punteada de AMBOS paneles# sigue 1/(1 - r0/2 ell), la ley del regulador DESPLAZADO, pese a que el# panel (b) es el suavizado. Aplicando la ec. (10) a la ec. (5) el resultado# correcto seria 1/(1 - r0/6 ell). Verificar en el PDF antes de citar.dig = np.load(RAIZ/"data/fig2_digitalizada.npz")fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))for ax, pan, tit in ((axes[0], "a", "Fig. 2(a) - desplazado"),                     (axes[1], "b", "Fig. 2(b) - suavizado")):    xd, sup, inf = dig[f"{pan}_x"], dig[f"{pan}_sup"], dig[f"{pan}_inf"]    libre = (sup < 0.80) & (inf > -1.22)         # no recortado por el marco    ax.fill_between(xd[libre], inf[libre], sup[libre], alpha=.2,                    label="region estable (paper)")    xx = np.linspace(2.2, 20, 400)    ax.plot(xx, 1/(1 - xx/2), lw=2, label=r"$1/(1-r_0/2\ell)$ desplazado")    xx6 = np.linspace(6.2, 20, 400)    ax.plot(xx6, 1/(1 - xx6/6), lw=2, ls="--", label=r"$1/(1-r_0/6\ell)$ suavizado")    ax.axhline(0, color="k", lw=.8)    ax.set_xlim(0, 21); ax.set_ylim(-1.4, 0.9)    ax.set_xlabel(r"$r_0/\ell$"); ax.set_ylabel("$m/m_B$"); ax.set_title(tit)    ax.legend(fontsize=8)plt.tight_layout(); plt.show()